In [1]:
import os
import dagshub
import mlflow
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import L1, L2, L1L2
from tensorflow.keras.datasets import mnist

# Conectar con DagsHub (Reemplaza con tus datos reales)
dagshub.init(repo_owner='bautistammarcos', repo_name='Red-Densa-MNIST', mlflow=True)
mlflow.set_experiment("MNIST_Regularization")


/Users/marcosbautista/uni/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as bautistammarcos

Initialized MLflow to track repo "bautistammarcos/Red-Densa-MNIST"

Repository bautistammarcos/Red-Densa-MNIST initialized!

2026/09/13 23:10:01 INFO mlflow.tracking.fluent: Experiment with name 'MNIST_Regularization' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/bcf375e7df7648da81c2e70c445dbdf4', creation_time=1789362602078, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1789362602078, lifecycle_stage='active', name='MNIST_Regularization', tags={}, trace_location=None, workspace='default'>

In [2]:
#Mnist 
(x_train,y_train),(x_test,y_test) = mnist.load_data()

#normalizando 
x_train = x_train.astype("float32")/255.0 
x_test = x_test.astype("float32")/255.0 

#vectorizando 
x_train = x_train.reshape(-1,28*28)
x_test = x_test.reshape(-1,28*28)

In [3]:
best_params = {
    'n_layers': 1,
    'units_0': 512,
    'activation': 'relu',
    'optimizer': 'adam',
    'learning_rate': 0.0012401052341003264
}

In [4]:
def build_model(params, regularizer=None, dropout_rate=None):
    model = Sequential()

    # Capa 0 (Entrada + Primera Capa Oculta)
    model.add(
        Dense(
            params['units_0'],
            activation=params['activation'],
            kernel_regularizer=regularizer,
            input_shape=(784,)
        )
    )
    if dropout_rate:
        model.add(Dropout(dropout_rate))

    # Capas ocultas subsecuentes segun best_params
    for i in range(1, params['n_layers']):
        model.add(
            Dense(
                params[f'units_{i}'],
                activation=params['activation'],
                kernel_regularizer=regularizer
            )
        )
        if dropout_rate:
            model.add(Dropout(dropout_rate))

    # Capa de salida
    model.add(Dense(10, activation="softmax"))

    # Optimizador segun best_params
    optimizer_cls = {
        "adam": tf.keras.optimizers.Adam,
        "rmsprop": tf.keras.optimizers.RMSprop,
        "sgd": tf.keras.optimizers.SGD
    }[params['optimizer']]

    optimizer = optimizer_cls(learning_rate=params['learning_rate'])

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [6]:
experiments = {
    "Base": {},
    "L1": {
        "regularizer": L1(1e-4),
        "reg_type": "L1",
        "l1_val": 1e-4
    },
    "L2": {
        "regularizer": L2(1e-4),
        "reg_type": "L2",
        "l2_val": 1e-4
    },
    "L1L2": {
        "regularizer": L1L2(l1=1e-5, l2=1e-4),
        "reg_type": "L1L2",
        "l1_val": 1e-5,
        "l2_val": 1e-4
    },
    "Dropout": {
        "dropout_rate": 0.3,
        "reg_type": "Dropout"
    },
    "Dropout+L1L2": {
        "regularizer": L1L2(l1=1e-5, l2=1e-4),
        "dropout_rate": 0.3,
        "reg_type": "Dropout+L1L2",
        "l1_val": 1e-5,
        "l2_val": 1e-4
    }
}

results = {}

for name, exp_config in experiments.items():
    print(f"\n{'='*50}")
    print(f"Entrenando experimento: {name}")
    print(f"{'='*50}")

    with mlflow.start_run(run_name=name):
        # 1. Registrar hiperparámetros del experimento
        mlflow.log_params(best_params)
        mlflow.log_param("regularization_name", name)
        mlflow.log_param("dropout_rate", exp_config.get("dropout_rate", 0.0))
        mlflow.log_param("reg_type", exp_config.get("reg_type", "None"))

        # 2. Construir y entrenar modelo
        model = build_model(
            best_params,
            regularizer=exp_config.get("regularizer"),
            dropout_rate=exp_config.get("dropout_rate")
        )

        history = model.fit(
            x_train,
            y_train,
            validation_split=0.2,
            epochs=20,
            batch_size=128,
            verbose=1
        )

        # 3. Registrar métricas época por época en MLflow
        for epoch in range(len(history.history['loss'])):
            mlflow.log_metric("train_loss", history.history['loss'][epoch], step=epoch)
            mlflow.log_metric("val_loss", history.history['val_loss'][epoch], step=epoch)
            mlflow.log_metric("train_accuracy", history.history['accuracy'][epoch], step=epoch)
            mlflow.log_metric("val_accuracy", history.history['val_accuracy'][epoch], step=epoch)

        # 4. Evaluación en conjunto de prueba (Test)
        test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

        train_acc = history.history["accuracy"][-1]
        val_acc = history.history["val_accuracy"][-1]
        train_loss = history.history["loss"][-1]
        val_loss = history.history["val_loss"][-1]
        gap = train_acc - val_acc

        # Registrar métricas finales
        mlflow.log_metric("test_accuracy", test_acc)
        mlflow.log_metric("test_loss", test_loss)
        mlflow.log_metric("train_val_gap", gap)

        results[name] = {
            "Train Accuracy": train_acc,
            "Validation Accuracy": val_acc,
            "Test Accuracy": test_acc,
            "Train Loss": train_loss,
            "Validation Loss": val_loss,
            "Test Loss": test_loss,
            "Gap": gap
        }

        print(f"Test Accuracy: {test_acc:.4f} | Gap (Train - Val Acc): {gap:.4f}")


Entrenando experimento: Base
Epoch 1/20


/Users/marcosbautista/uni/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9176 - loss: 0.2808 - val_accuracy: 0.9581 - val_loss: 0.1443
Epoch 2/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9669 - loss: 0.1127 - val_accuracy: 0.9679 - val_loss: 0.1037
Epoch 3/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9794 - loss: 0.0712 - val_accuracy: 0.9711 - val_loss: 0.0971
Epoch 4/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9854 - loss: 0.0506 - val_accuracy: 0.9747 - val_loss: 0.0829
Epoch 5/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9892 - loss: 0.0371 - val_accuracy: 0.9768 - val_loss: 0.0806
Epoch 6/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9919 - loss: 0.0276 - val_accuracy: 0.9762 - val_loss: 0.0818
Epoch 7/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9948 - loss: 0.0202 - val_accuracy: 0.9757 - val_loss: 0.0868
Epoch 8/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9967 - loss: 0.0136 - val_accuracy: 0.9791 - val_

In [7]:
results = {}

experiments = {
    "Base": {},
    "L1": {
        "regularizer": L1(1e-4),
        "reg_type": "L1",
        "reg_val": 1e-4
    },
    "L2": {
        "regularizer": L2(1e-4),
        "reg_type": "L2",
        "reg_val": 1e-4
    },
    "L1L2": {
        "regularizer": L1L2(l1=1e-5, l2=1e-4),
        "reg_type": "L1_L2",
        "reg_val": "l1=1e-5, l2=1e-4"
    },
    "Dropout": {
        "dropout_rate": 0.3,
        "reg_type": "Dropout",
        "dropout_val": 0.3
    },
    "Dropout+L1L2": {
        "regularizer": L1L2(l1=1e-5, l2=1e-4),
        "dropout_rate": 0.3,
        "reg_type": "Dropout + L1_L2",
        "dropout_val": 0.3,
        "reg_val": "l1=1e-5, l2=1e-4"
    }
}

epochs = 20

for name, exp_config in experiments.items():
    print(f"\n{'='*50}")
    print(f"Entrenando experimento: {name}")
    print(f"{'='*50}")

    with mlflow.start_run(run_name=f"Reg_{name}"):
        
        # 1. Registrar hiperparámetros
        mlflow.log_params(best_params)
        mlflow.log_param("experiment_name", name)
        mlflow.log_param("regularizer_type", exp_config.get("reg_type", "None"))
        mlflow.log_param("dropout_rate", exp_config.get("dropout_rate", 0.0))

        # 2. Entrenar el modelo
        model = build_model(
            params=best_params,
            regularizer=exp_config.get("regularizer"),
            dropout_rate=exp_config.get("dropout_rate")
        )

        history = model.fit(
            x_train,
            y_train,
            validation_split=0.2,
            epochs=epochs,
            batch_size=128,
            verbose=1
        )

        # 3. OPTIMIZACIÓN: Preparar todas las métricas en listas para enviarlas juntas
        metrics_payload = {}
        for ep in range(epochs):
            mlflow.log_metrics({
                "train_loss": history.history["loss"][ep],
                "val_loss": history.history["val_loss"][ep],
                "train_accuracy": history.history["accuracy"][ep],
                "val_accuracy": history.history["val_accuracy"][ep]
            }, step=ep)

        # 4. Evaluación final en test set
        test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

        train_acc = history.history["accuracy"][-1]
        val_acc = history.history["val_accuracy"][-1]
        train_loss = history.history["loss"][-1]
        val_loss = history.history["val_loss"][-1]
        gap = train_acc - val_acc

        # Registrar resumen final
        mlflow.log_metrics({
            "final_test_accuracy": test_acc,
            "final_test_loss": test_loss,
            "accuracy_gap": gap
        })

        results[name] = {
            "Train Accuracy": train_acc,
            "Validation Accuracy": val_acc,
            "Test Accuracy": test_acc,
            "Train Loss": train_loss,
            "Validation Loss": val_loss,
            "Test Loss": test_loss,
            "Gap": gap
        }

        print(f"Test Accuracy: {test_acc:.4f} | Gap (Train - Val): {gap:.4f}")


Entrenando experimento: Base
Epoch 1/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9209 - loss: 0.2793 - val_accuracy: 0.9539 - val_loss: 0.1573
Epoch 2/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9663 - loss: 0.1153 - val_accuracy: 0.9704 - val_loss: 0.1006
Epoch 3/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9778 - loss: 0.0741 - val_accuracy: 0.9732 - val_loss: 0.0886
Epoch 4/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9848 - loss: 0.0509 - val_accuracy: 0.9762 - val_loss: 0.0804
Epoch 5/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9897 - loss: 0.0370 - val_accuracy: 0.9743 - val_loss: 0.0829
Epoch 6/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9927 - loss: 0.0266 - val_accuracy: 0.9747 - val_loss: 0.0855
Epoch 7/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9947 - loss: 0.0203 - val_accuracy: 0.9774 - val_loss: 0.0760
Epoch 8/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9965 - l